# WhatsApp Chat Parser
## Consolidate 2600+ WhatsApp TXT exports into a single DataFrame

This notebook parses WhatsApp chat exports in TXT format and creates a consolidated DataFrame for analytics.

In [ ]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

## Configuration
### Specify the folder containing your WhatsApp TXT files

In [ ]:
# Ask user for folder path
folder_path = input("Enter the folder path containing WhatsApp TXT files: ")
folder = Path(folder_path)

if not folder.exists():
    raise ValueError(f"Folder does not exist: {folder_path}")

# Find all TXT files
txt_files = list(folder.glob('*.txt'))
print(f"Found {len(txt_files)} TXT files to process")

## Parser Function
### Based on observed patterns from sample file:
- Line 1: Device header (e.g., "Samsung SM-A065M(+5215532236520)")
- Line 2: Separator line
- Messages follow pattern:
  - Timestamp: `YYYY/MM/DD HH:MM`
  - Sender: `+5215532236520:` or `-:`
  - Message text (can be multiline)
  - Blank line separator (optional)

In [ ]:
def parse_whatsapp_txt(file_path):    """    Parse a WhatsApp TXT export file into structured records.        Returns:        list of dict: Each dict contains message data    """    messages = []        try:        # Read file with UTF-8 encoding to preserve emojis        with open(file_path, 'r', encoding='utf-8-sig') as f:            lines = f.readlines()                if len(lines) < 3:            return messages                # Extract device info and chat ID from header        device_header = lines[0].strip()        chat_id = None                # Extract phone number from header like "Samsung SM-A065M(+5215532236520)"        header_match = re.search(r'\(([+\d]+)\)', device_header)        if header_match:            chat_id = header_match.group(1)        else:            # Fallback to filename            chat_id = file_path.stem                # Timestamp pattern: YYYY/MM/DD HH:MM        timestamp_pattern = re.compile(r'^(\d{4}/\d{2}/\d{2} \d{2}:\d{2})$')                i = 2  # Start after header and separator        message_index = 0                while i < len(lines):            line = lines[i].strip()                        # Check if this is a timestamp line            timestamp_match = timestamp_pattern.match(line)                        if timestamp_match:                timestamp_str = timestamp_match.group(1)                                # Next line should be sender                if i + 1 < len(lines):                    sender_line = lines[i + 1].strip()                                        # Check if sender line ends with ':'                    if sender_line.endswith(':'):                        sender = sender_line[:-1]  # Remove trailing ':'                                                # Determine message direction                        if sender == '-':                            direction = 'outgoing'                            sender_display = 'self'                        else:                            direction = 'incoming'                            sender_display = sender                                                # Collect message text (may be multiline)                        message_lines = []                        j = i + 2                                                # Read lines until we hit next timestamp or end of file                        while j < len(lines):                            next_line = lines[j].strip()                                                        # Stop if we hit another timestamp                            if timestamp_pattern.match(next_line):                                break                                                        # Add non-empty lines to message                            if next_line:                                message_lines.append(next_line)                                                        j += 1                                                message_text = ' '.join(message_lines)                                                # Parse timestamp                        try:                            timestamp = datetime.strptime(timestamp_str, '%Y/%m/%d %H:%M')                        except:                            timestamp = None                                                # Create message record                        messages.append({                            'chat_id': chat_id,                            'file_name': file_path.name,                            'message_index': message_index,                            'timestamp': timestamp,                            'date': timestamp.date() if timestamp else None,                            'time': timestamp.time() if timestamp else None,                            'direction': direction,                            'sender': sender_display,                            'sender_raw': sender,                            'message': message_text,                            'message_length': len(message_text),                            'word_count': len(message_text.split()) if message_text else 0                        })                                                message_index += 1                        i = j  # Move to next timestamp                        continue                        i += 1        except Exception as e:        print(f"Error parsing {file_path.name}: {str(e)}")        return messages

## Test Parser on Sample File
### Let's test the parser on one file first to verify it works correctly

In [ ]:
# Test on first file
if txt_files:
    test_file = txt_files[0]
    print(f"Testing parser on: {test_file.name}")
    test_messages = parse_whatsapp_txt(test_file)
    
    if test_messages:
        test_df = pd.DataFrame(test_messages)
        print(f"\nParsed {len(test_messages)} messages")
        print("\nFirst few messages:")
        display(test_df.head(10))
        
        print("\nDataFrame Info:")
        print(test_df.info())
        
        print("\nSample message:")
        print(test_df.iloc[0]['message'])
    else:
        print("No messages parsed. Check file format.")

## Process All Files
### Parse all 2600+ TXT files with progress tracking

In [ ]:
# Process all files
all_messages = []
failed_files = []

print(f"Processing {len(txt_files)} files...\n")

for file_path in tqdm(txt_files, desc="Parsing files"):
    messages = parse_whatsapp_txt(file_path)
    
    if messages:
        all_messages.extend(messages)
    else:
        failed_files.append(file_path.name)

print(f"\n✓ Successfully parsed {len(txt_files) - len(failed_files)} files")
print(f"✓ Total messages extracted: {len(all_messages):,}")

if failed_files:
    print(f"\n⚠ Failed to parse {len(failed_files)} files:")
    for f in failed_files[:10]:  # Show first 10
        print(f"  - {f}")
    if len(failed_files) > 10:
        print(f"  ... and {len(failed_files) - 10} more")

## Create Consolidated DataFrame

In [ ]:
# Create DataFrame
df = pd.DataFrame(all_messages)

print(f"DataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## Data Overview and Statistics

In [ ]:
# Display first rows
print("First 10 messages:")
display(df.head(10))

In [ ]:
# Basic statistics
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

print(f"\nTotal messages: {len(df):,}")
print(f"Total chats: {df['chat_id'].nunique():,}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

print("\nMessage direction:")
print(df['direction'].value_counts())

print("\nTop 10 most active chats:")
print(df['chat_id'].value_counts().head(10))

print("\nAverage message length:")
print(df.groupby('direction')['message_length'].mean())

print("\nAverage word count:")
print(df.groupby('direction')['word_count'].mean())

## Save DataFrame

In [ ]:
# Save to CSV
output_csv = folder / 'whatsapp_consolidated.csv'
df.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"✓ Saved to CSV: {output_csv}")

# Save to Parquet (more efficient for large datasets)
output_parquet = folder / 'whatsapp_consolidated.parquet'
df.to_parquet(output_parquet, index=False)
print(f"✓ Saved to Parquet: {output_parquet}")

# Save to Excel (first 1M rows only due to Excel limits)
if len(df) <= 1048576:
    output_excel = folder / 'whatsapp_consolidated.xlsx'
    df.to_excel(output_excel, index=False)
    print(f"✓ Saved to Excel: {output_excel}")
else:
    print(f"⚠ Dataset too large for Excel ({len(df):,} rows). Use CSV or Parquet instead.")

## Quick Analytics Examples

In [ ]:
# Messages per day
df['date'] = pd.to_datetime(df['date'])
messages_per_day = df.groupby('date').size()

print("Messages per day (last 10 days):")
print(messages_per_day.tail(10))

In [ ]:
# Messages by hour of day
df['hour'] = df['time'].apply(lambda x: x.hour if x else None)
messages_by_hour = df.groupby('hour').size()

print("Messages by hour of day:")
print(messages_by_hour)

In [ ]:
# Response time analysis (time between incoming and outgoing messages)
df_sorted = df.sort_values(['chat_id', 'timestamp'])
df_sorted['time_diff'] = df_sorted.groupby('chat_id')['timestamp'].diff()

# Filter for outgoing messages that follow incoming messages
df_sorted['prev_direction'] = df_sorted.groupby('chat_id')['direction'].shift(1)
response_times = df_sorted[
    (df_sorted['direction'] == 'outgoing') & 
    (df_sorted['prev_direction'] == 'incoming')
]['time_diff']

print("Response time statistics:")
print(f"Median response time: {response_times.median()}")
print(f"Mean response time: {response_times.mean()}")

## DataFrame is ready for analysis!

### Available columns:
- `chat_id`: Unique identifier for each chat
- `file_name`: Original filename
- `message_index`: Sequential message number within each chat
- `timestamp`: Full datetime of message
- `date`: Date only
- `time`: Time only
- `direction`: 'incoming' or 'outgoing'
- `sender`: Display name ('self' for outgoing, phone number for incoming)
- `sender_raw`: Raw sender string from file
- `message`: Full message text
- `message_length`: Character count
- `word_count`: Word count

### You can now perform any analytics on the `df` DataFrame!